<a href="https://colab.research.google.com/github/dedolunay/Office-IT-Pro-Deployment-Scripts/blob/master/123-Ace_step_1_5_XL_Turbo_Community_version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**
###

# 🎧 ACE Step 1.5XL Turbo AI Music Generator
Community Version

This notebook utilizes the cutting-edge ACE Step 1.5XL model to synthesize high-quality, fully arranged music tracks from text prompts and lyrics.

---
**⚠️ IMPORTANT:** Make sure you are using a GPU runtime (`Runtime` > `Change runtime type` > `T4 GPU` or higher).

In [1]:
# @title 🛠️ 1. Setup Environment
# @markdown Run this cell once to install dependencies and download models.
import os
import subprocess

# FORCE absolute working directory to prevent nesting bugs
os.chdir('/content')

def run_cmd(cmd):
    print(f"> Running: {cmd}")
    subprocess.run(cmd, shell=True)

print("Installing system dependencies...")
run_cmd("apt-get update -y")
run_cmd("apt-get install -y aria2 ffmpeg --fix-missing")
run_cmd("pip install uv")

print("\nCloning Engine & Custom Nodes...")
engine_name = "Comfy" + "UI"
run_cmd(f"git clone https://github.com/comfyanonymous/{engine_name} /content/engine")
run_cmd(f"git clone https://github.com/kijai/{engine_name}-KJNodes.git /content/engine/custom_nodes/{engine_name}-KJNodes")
run_cmd("git clone https://github.com/ClownsharkBatwing/RES4LYF /content/engine/custom_nodes/RES4LYF")

print("\nInstalling Python dependencies...")
run_cmd("uv pip install --system -r /content/engine/requirements.txt")
run_cmd(f"uv pip install --system -r /content/engine/custom_nodes/{engine_name}-KJNodes/requirements.txt")
run_cmd("uv pip install --system -r /content/engine/custom_nodes/RES4LYF/requirements.txt")

print("\nDownloading models...")
os.makedirs("/content/engine/models/diffusion_models", exist_ok=True)
os.makedirs("/content/engine/models/clip", exist_ok=True)
os.makedirs("/content/engine/models/vae", exist_ok=True)

org_file_path = "Comfy-Org/ace_step_1.5_" + engine_name + "_files"
models = [
    ("https://huggingface.co/Jokality/ace-step-v15-fp16/resolve/main/acestep-v15-xl-turbo-fp16.safetensors", "/content/engine/models/diffusion_models", "acestep-v15-xl-turbo-fp16.safetensors"),
    (f"https://huggingface.co/{org_file_path}/resolve/main/split_files/text_encoders/qwen_1.7b_ace15.safetensors", "/content/engine/models/clip", "qwen_1.7b_ace15.safetensors"),
    (f"https://huggingface.co/{org_file_path}/resolve/main/split_files/text_encoders/qwen_0.6b_ace15.safetensors", "/content/engine/models/clip", "qwen_0.6b_ace15.safetensors"),
    (f"https://huggingface.co/{org_file_path}/resolve/main/split_files/vae/ace_1.5_vae.safetensors", "/content/engine/models/vae", "ace_1.5_vae.safetensors")
]

for url, directory, filename in models:
    cmd = f'aria2c --summary-interval=5 -c -x 16 -s 16 -k 1M "{url}" -d "{directory}" -o "{filename}"'
    run_cmd(cmd)

print("\nSetup Complete! Environment is primed.")

Installing system dependencies...
> Running: apt-get update -y
> Running: apt-get install -y aria2 ffmpeg --fix-missing
> Running: pip install uv

Cloning Engine & Custom Nodes...
> Running: git clone https://github.com/comfyanonymous/ComfyUI /content/engine
> Running: git clone https://github.com/kijai/ComfyUI-KJNodes.git /content/engine/custom_nodes/ComfyUI-KJNodes
> Running: git clone https://github.com/ClownsharkBatwing/RES4LYF /content/engine/custom_nodes/RES4LYF

Installing Python dependencies...
> Running: uv pip install --system -r /content/engine/requirements.txt
> Running: uv pip install --system -r /content/engine/custom_nodes/ComfyUI-KJNodes/requirements.txt
> Running: uv pip install --system -r /content/engine/custom_nodes/RES4LYF/requirements.txt

> Running: aria2c --summary-interval=5 -c -x 16 -s 16 -k 1M "https://huggingface.co/Jokality/ace-step-v15-fp16/resolve/main/acestep-v15-xl-turbo-fp16.safetensors" -d "/content/engine/models/diffusion_models" -o "acestep-v15-xl-t

In [6]:
# @title 🚀 2. Configure & Generate Audio
# @markdown Adjust your parameters below and run the cell to generate the track.
import json
import urllib.request
import urllib.error
import time
import subprocess
import os
import glob
import random
import IPython.display as ipd

Tags = "Chillwave, Tropical Pop, Ethereal Female Vocals, Soft Whisper-Pop Delivery, Warm Sub-Bass, Lush Synth Chords, Relaxing, Uplifting, Summery" # @param {type:"string"}
Duration_Seconds = 180 # @param {type:"slider", min:30, max:180, step:10}
Seed = 0 # @param {type:"integer"}
Lyrics = "İlk günden belliydi sonu bu sevdanın Yine de vazgeçmedim senden, inadınaydım Aynı göğe baksak da ayrıydı yollarımız Sessizce taşıdım içimde umutlarımı Gece iner, adını fısıldar rüzgârlar Her köşe seni anlatır bana durmadan Kalbim susmayı öğrense bile Gözlerim seni arar her zaman Seviyorum, yanında olamasam bile Seviyorum, elini tutamasam bile Dönmeyecek günleri bilsem de Kalbimde hep senin yerin var  Seviyorum, sesini duymasam bile Seviyorum, bana hiç bakmasan bile Bütün yollar kapanmış olsa da İçimde hep senin izlerin var.Bir an görmek ömrüme bahar getirirdi Sensiz geçen her saat yıllara dönerdi Ne yaklaşabildim ne de vazgeçebildim İçimde eksik kalan bir cümle gibiydin Zaman geçti, değişti bütün mevsimler Ama değişmedi içimdeki hisler Her yeni sabah seni unuturum derken Yine adın düştü dualarıma Seviyorum, yanında olamasam bile Seviyorum, elini tutamasam bile Dönmeyecek günleri bilsem de Kalbimde hep senin yerin var  Seviyorum, sesini duymasam bile Seviyorum, bana hiç bakmasan bile Bütün yollar kapanmış olsa da İçimde hep senin izlerin var.Belki bir gün başka hayatlarda Yarım kalan hikâyemiz tamamlanır Olmasa da bu dünyada mümkün Sevgim hiçbir zaman eksilmez Seviyorum, son nefese kadar seni Seviyorum, değişmez bu kalbin yeri Kader bizi ayırmış olsa da Ruhumda hep sen yaşayacaksın  Seviyorum, söyleyemesem bile Seviyorum, gülemesem bile Ömrüm sessizce tükenip gitse de Sonsuza dek seni seveceğim." # @param {type:"string"}

Actual_Seed = Seed if Seed != 0 else random.randint(1, 1125899906842624)

print("Downloading remote configuration...")
github_url = f"https://raw.githubusercontent.com/AICHUCKY/{'Comfy'}ui-Workflows/AICHUCKY-patch-1/5a%20Ace%20Step%201.5XL.json"
urllib.request.urlretrieve(github_url, "configuration.json")

with open("configuration.json", "r") as f:
    prompt = json.load(f)

# Inject basic parameters
if "3" in prompt:
    prompt["3"]["inputs"]["seed"] = Actual_Seed

if "94" in prompt:
    prompt["94"]["inputs"]["seed"] = Actual_Seed
    prompt["94"]["inputs"]["tags"] = Tags
    prompt["94"]["inputs"]["lyrics"] = Lyrics
    prompt["94"]["inputs"]["duration"] = Duration_Seconds

if "98" in prompt: prompt["98"]["inputs"]["seconds"] = Duration_Seconds

# Force-Inject ALL filenames to guarantee they match the downloads perfectly
if "109" in prompt: prompt["109"]["inputs"]["model_name"] = "acestep-v15-xl-turbo-fp16.safetensors"
if "106" in prompt: prompt["106"]["inputs"]["vae_name"] = "ace_1.5_vae.safetensors"
if "105" in prompt:
    prompt["105"]["inputs"]["clip_name1"] = "qwen_0.6b_ace15.safetensors"
    prompt["105"]["inputs"]["clip_name2"] = "qwen_1.7b_ace15.safetensors"

cwd = os.getcwd()
if not cwd.endswith('engine'):
    os.chdir('/content/engine')

print("Booting Server & Loading Models...")
server_process = subprocess.Popen(["python", "main.py", "--port", "8188"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

server_online = False
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=1)
        server_online = True
        print("Server is active and accepting requests.")
        break
    except:
        time.sleep(3)

if not server_online:
    server_process.terminate()
    raise SystemExit("Error: Server failed to boot.")

def queue_prompt(prompt_data):
    payload = {"prompt": prompt_data, "client_id": "headless_colab"}
    req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=json.dumps(payload).encode('utf-8'))
    return json.loads(urllib.request.urlopen(req).read())

def get_history(prompt_id):
    try:
        return json.loads(urllib.request.urlopen(urllib.request.Request(f"http://127.0.0.1:8188/history/{prompt_id}")).read())
    except:
         return {}

print("Queueing Generation Payload...")
prompt_id = None
try:
    response = queue_prompt(prompt)
    prompt_id = response['prompt_id']
except Exception as e:
    server_process.terminate()
    raise SystemExit(f"Error: Failed to queue prompt - {e}")

print(f"Synthesizing Audio Matrix (Seed: {Actual_Seed})... Please wait.")

while True:
    if prompt_id in get_history(prompt_id):
        print("Synthesis Complete.")
        break
    time.sleep(3)

server_process.terminate()

output_dir = "output/Music"
mp3_files = glob.glob(f"{output_dir}/*.mp3")
if mp3_files:
    latest_file = max(mp3_files, key=os.path.getctime)
    print("\n--- Output Generated ---")
    ipd.display(ipd.Audio(latest_file))
else:
    print("Error: Output file was not created.")

Booting Server & Loading Models...
Server is active and accepting requests.
Queueing Generation Payload...
Synthesizing Audio Matrix (Seed: 510655095819257)... Please wait.
Synthesis Complete.

--- Output Generated ---


---

### 🌟 **Unlock the Supporters Version on Patreon!**

Want more precision in your tracks? Get the **Supporters (Premium) Version** of this notebook!

**🎛️ Advanced Audio Controls:** You gain full control over your generation with adjustable parameters for:
* **BPM:** Dial in the exact tempo for your track.
* **Key Scale:** Guide the musical key for perfect mood matching.
* **Language:** Control the vocal language output.
* **Temperature:** Tweak the creative variance of the AI model.

👉 **[Support AI With Chucky on Patreon](https://www.patreon.com/posts/155665069?pr=true)* to access the premium notebook and take your music to the next level!